# GEARS Training - GSE152988 CRISPRi Dataset

Following the official GEARS tutorial structure.

Reference: https://github.com/snap-stanford/GEARS

## Step 1: Imports

In [ ]:
import scanpy as sc
import numpy as np
from gears import PertData, GEARS
import pandas as pd

print("Imports successful")

c:\Users\bfong\Desktop\Yume\Gear\venv_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful


## Step 1.5: Check GPU Availability

In [ ]:
import torch

# Check GPU availability
if torch.cuda.is_available():
    device = 'cuda:0'
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Using device: {device}")
elif torch.backends.mps.is_available():
    device = 'mps'  # Apple Silicon GPU
    print(f"✓ Apple Silicon GPU available")
    print(f"  Using device: {device}")
else:
    device = 'cpu'
    print("No GPU detected, using CPU")
    print(f"  Using device: {device}")

print(f"\nPyTorch version: {torch.__version__}")

✓ GPU available: NVIDIA GeForce RTX 3060 Ti
  Using device: cuda:0

PyTorch version: 2.7.1+cu118


## Step 2: Load and Format Data

In [ ]:
# Load raw data and format for GEARS
adata = sc.read_h5ad('Datasets/Perturbation/GSE152988/TianKampmann2021_CRISPRi.h5ad')

# Add required columns
adata.var['gene_name'] = adata.var.index
adata.obs['condition'] = adata.obs['perturbation'].astype(str).replace('control', 'ctrl')

if 'celltype' in adata.obs.columns:
    adata.obs['cell_type'] = adata.obs['celltype']
else:
    adata.obs['cell_type'] = 'iPSC_neuron'

print(f"Loaded: {adata.shape[0]} cells x {adata.shape[1]} genes")

Loaded: 32300 cells x 33538 genes


## Step 2.5: Inspect Data (Optional)

In [ ]:
# Check what's in the data
adata
print(f"Shape: {adata.shape}")
print(f"\nobs columns: {list(adata.obs.columns)}")
print(f"\nFirst few cells:")
print(adata.obs.head())
print(f"\nCondition counts:")
print(adata.obs['condition'].value_counts())
print(f"\nTotal unique perturbations: {adata.obs['condition'].nunique()}")

Shape: (32300, 33538)

obs columns: ['guide_id', 'perturbation', 'tissue_type', 'celltype', 'cancer', 'disease', 'perturbation_type', 'organism', 'batch', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'percent_hemo', 'condition', 'cell_type']

First few cells:
                   guide_id perturbation tissue_type             celltype  \
AAACCCAAGGGTTGCA  CREBBP_i7       CREBBP     primary  iPSC-induced neuron   
AAACCCAAGGTAGCCA  SH3RF1_i2       SH3RF1     primary  iPSC-induced neuron   
AAACCCACAAACTAGA    TAF1_i5         TAF1     primary  iPSC-induced neuron   
AAACCCACAGAATTCC   UBTD2_i1        UBTD2     primary  iPSC-induced neuron   
AAACCCAGTAGCGTTT  FRMD4A_i2       FRMD4A     primary  iPSC-induced neuron   

                  cancer  disease perturbation_type organism batch  nperts  \
AAACCCAAGGGTTGCA   False  healthy            CRISPR    human     1       1   
AAACCCAAGGTAGCCA   False  healthy            CRISPR    human     1       1   
AAACCCACAAACTAGA   False 

## Step 3: Initialize PertData and Process Dataset

In [ ]:
# get data
pert_data = PertData('./data')

# process new dataset
pert_data.new_data_process(dataset_name='GSE152988_CRISPRi', adata=adata)

Downloading...
100%|██████████| 9.46M/9.46M [00:00<00:00, 12.6MiB/s]
Downloading...
100%|██████████| 559k/559k [00:00<00:00, 1.82MiB/s]
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 185/185 [1:10:22<00:00, 22.82s/it]
Done!
Saving new dataset pyg object at ./data\gse152988_crispri\data_pyg\cell_graphs.pkl
Done!


## Step 4: Load Dataset and train the model

In [ ]:
from gears import PertData, GEARS
import gears.pertdata as pertdata_module
import numpy as np
import os
import torch

# Fix buggy function
def fixed_filter_pert_in_go(condition, pert_names):
    if condition == 'ctrl':
        return True
    genes = condition.split('+')
    return all(gene in pert_names for gene in genes)

pertdata_module.filter_pert_in_go = fixed_filter_pert_in_go

# Create directories
os.makedirs('./data/gse152988_crispri/splits/data', exist_ok=True)
os.makedirs('./data/data/gse152988_crispri', exist_ok=True)

# Load data
pert_data = PertData('./data')
pert_data.load(data_path='./data/gse152988_crispri')

# Manual splits
all_perts = [p for p in pert_data.adata.obs['condition'].unique() if p != 'ctrl']
np.random.seed(1)
np.random.shuffle(all_perts)

n_total = len(all_perts)
pert_data.set2conditions = {
    'train': all_perts[:int(0.70*n_total)],
    'val': all_perts[int(0.70*n_total):int(0.85*n_total)],
    'test': all_perts[int(0.85*n_total):]
}

pert_data.split = 'custom'
pert_data.seed = 1
pert_data.train_gene_set_size = 0.75

print(f"Split: Train={len(pert_data.set2conditions['train'])}, Val={len(pert_data.set2conditions['val'])}, Test={len(pert_data.set2conditions['test'])}")

# Get dataloaders
pert_data.get_dataloader(batch_size=32, test_batch_size=128)

# Create empty graphs manually (valid but minimal)
num_genes = len(pert_data.gene_names)

# Empty edge indices (shape: [2, 0])
G_coexpress = torch.LongTensor(2, 0).to(device)
G_coexpress_weight = torch.Tensor(0).to(device)

# Train with pre-defined empty graphs
gears_model = GEARS(pert_data, device=device)
gears_model.model_initialize(
    hidden_size=64,
    G_coexpress=G_coexpress,
    G_coexpress_weight=G_coexpress_weight
)

gears_model.train(epochs=20)

Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
[]
Local copy of pyg dataset is detected. Loading...
Done!
Creating dataloaders....
Done!


Split: Train=128, Val=28, Test=28


Found local copy...
Start Training...
Epoch 1 Step 1 Train Loss: 8449996.0000
Epoch 1 Step 51 Train Loss: 16647491.0000
Epoch 1 Step 101 Train Loss: 15244825.0000
Epoch 1 Step 151 Train Loss: 32905640.0000
Epoch 1 Step 201 Train Loss: 6161983.5000
Epoch 1 Step 251 Train Loss: 42967692.0000
Epoch 1 Step 301 Train Loss: 22316476.0000
Epoch 1 Step 351 Train Loss: 3786040.7500
Epoch 1 Step 401 Train Loss: 13005115.0000
Epoch 1 Step 451 Train Loss: 7176703.0000
Epoch 1 Step 501 Train Loss: 4724781.0000
Epoch 1 Step 551 Train Loss: 44721804.0000
Epoch 1 Step 601 Train Loss: 10158371.0000
Epoch 1 Step 651 Train Loss: 14950472.0000
Epoch 1 Step 701 Train Loss: 17938066.0000
Epoch 1: Train Overall MSE: 0.1257 Validation Overall MSE: 0.1465. 
Train Top 20 DE MSE: 1.2613 Validation Top 20 DE MSE: 0.1648. 
Epoch 2 Step 1 Train Loss: 12941250.0000
Epoch 2 Step 51 Train Loss: 33526400.0000
Epoch 2 Step 101 Train Loss: 12630458.0000
Epoch 2 Step 151 Train Loss: 167642816.0000
Epoch 2 Step 201 Train L

In [ ]:
from gears import PertData, GEARS
import gears.pertdata as pertdata_module
import numpy as np
import os
import torch

# Fix buggy function
def fixed_filter_pert_in_go(condition, pert_names):
    if condition == 'ctrl':
        return True
    genes = condition.split('+')
    return all(gene in pert_names for gene in genes)

pertdata_module.filter_pert_in_go = fixed_filter_pert_in_go

# Create directories
os.makedirs('./data/gse152988_crispri/splits/data', exist_ok=True)
os.makedirs('./data/data/gse152988_crispri', exist_ok=True)

# Load data
pert_data = PertData('./data')
pert_data.load(data_path='./data/gse152988_crispri')

# Manual splits
all_perts = [p for p in pert_data.adata.obs['condition'].unique() if p != 'ctrl']
np.random.seed(1)
np.random.shuffle(all_perts)

n_total = len(all_perts)
pert_data.set2conditions = {
    'train': all_perts[:int(0.70*n_total)],
    'val': all_perts[int(0.70*n_total):int(0.85*n_total)],
    'test': all_perts[int(0.85*n_total):]
}

pert_data.split = 'custom'
pert_data.seed = 1
pert_data.train_gene_set_size = 0.75

print(f"Split: Train={len(pert_data.set2conditions['train'])}, Val={len(pert_data.set2conditions['val'])}, Test={len(pert_data.set2conditions['test'])}")

# Get dataloaders
pert_data.get_dataloader(batch_size=32, test_batch_size=128)

# Create empty graphs manually (valid but minimal)
num_genes = len(pert_data.gene_names)

# Empty edge indices (shape: [2, 0])
G_coexpress = torch.LongTensor(2, 0).to(device)
G_coexpress_weight = torch.Tensor(0).to(device)

# Train with pre-defined empty graphs
gears_model = GEARS(pert_data, device=device)
gears_model.model_initialize(
    hidden_size=64,
    G_coexpress=G_coexpress,
    G_coexpress_weight=G_coexpress_weight
)

gears_model.train(epochs=20)

Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
[]
Local copy of pyg dataset is detected. Loading...
Done!
Creating dataloaders....
Done!


Split: Train=128, Val=28, Test=28


Found local copy...
Start Training...
Epoch 1 Step 1 Train Loss: 8449996.0000
Epoch 1 Step 51 Train Loss: 16647491.0000
Epoch 1 Step 101 Train Loss: 15244825.0000
Epoch 1 Step 151 Train Loss: 32905640.0000
Epoch 1 Step 201 Train Loss: 6161983.5000
Epoch 1 Step 251 Train Loss: 42967692.0000
Epoch 1 Step 301 Train Loss: 22316476.0000
Epoch 1 Step 351 Train Loss: 3786040.7500
Epoch 1 Step 401 Train Loss: 13005115.0000
Epoch 1 Step 451 Train Loss: 7176703.0000
Epoch 1 Step 501 Train Loss: 4724781.0000
Epoch 1 Step 551 Train Loss: 44721804.0000
Epoch 1 Step 601 Train Loss: 10158371.0000
Epoch 1 Step 651 Train Loss: 14950472.0000
Epoch 1 Step 701 Train Loss: 17938066.0000
Epoch 1: Train Overall MSE: 0.1257 Validation Overall MSE: 0.1465. 
Train Top 20 DE MSE: 1.2613 Validation Top 20 DE MSE: 0.1648. 
Epoch 2 Step 1 Train Loss: 12941250.0000
Epoch 2 Step 51 Train Loss: 33526400.0000
Epoch 2 Step 101 Train Loss: 12630458.0000
Epoch 2 Step 151 Train Loss: 167642816.0000
Epoch 2 Step 201 Train L

In [ ]:
# from gears import PertData, GEARS
# import os

# # Create all directories GEARS might need
# os.makedirs('./data/norman/splits/data', exist_ok=True)
# os.makedirs('./data/data/norman', exist_ok=True)

# pert_data = PertData('./data')
# pert_data.load(data_name='norman')
# pert_data.prepare_split(split='simulation', seed=1)
# pert_data.get_dataloader(batch_size=32, test_batch_size=128)

# gears_model = GEARS(pert_data, device=device)
# gears_model.model_initialize(hidden_size=64)
# gears_model.train(epochs=5)

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:3
combo_seen1:53
combo_seen2:18
unseen_single:0
Done!
Creating dataloaders....
Done!


here1


Downloading...
100%|██████████| 60.7M/60.7M [00:02<00:00, 20.8MiB/s]
Extracting tar file...
Done!
Start Training...
Epoch 1 Step 1 Train Loss: 0.5745
Epoch 1 Step 51 Train Loss: 0.4545
Epoch 1 Step 101 Train Loss: 0.4902
Epoch 1 Step 151 Train Loss: 0.5082
Epoch 1 Step 201 Train Loss: 0.4863
Epoch 1 Step 251 Train Loss: 0.5313
Epoch 1 Step 301 Train Loss: 0.5589
Epoch 1 Step 351 Train Loss: 0.5270
Epoch 1 Step 401 Train Loss: 0.5004
Epoch 1 Step 451 Train Loss: 0.5668
Epoch 1 Step 501 Train Loss: 0.5571
Epoch 1 Step 551 Train Loss: 0.4493
Epoch 1: Train Overall MSE: 0.0067 Validation Overall MSE: 0.0084. 
Train Top 20 DE MSE: 0.1353 Validation Top 20 DE MSE: 0.3037. 
Epoch 2 Step 1 Train Loss: 0.4896
Epoch 2 Step 51 Train Loss: 0.5009
Epoch 2 Step 101 Train Loss: 0.4940
Epoch 2 Step 151 Train Loss: 0.4725
Epoch 2 Step 201 Train Loss: 0.4822
Epoch 2 Step 251 Train Loss: 0.5010
Epoch 2 Step 301 Train Loss: 0.5631
Epoch 2 Step 351 Train Loss: 0.4512
Epoch 2 Step 401 Train Loss: 0.5019
Epo

In [ ]:
from gears import PertData, GEARS
import gears.pertdata as pertdata_module
import numpy as np
import os
import torch

# Fix buggy function
def fixed_filter_pert_in_go(condition, pert_names):
    if condition == 'ctrl':
        return True
    genes = condition.split('+')
    return all(gene in pert_names for gene in genes)

pertdata_module.filter_pert_in_go = fixed_filter_pert_in_go

# Create directories
os.makedirs('./data/gse152988_crispri/splits/data', exist_ok=True)
os.makedirs('./data/data/gse152988_crispri', exist_ok=True)

# Load data
pert_data = PertData('./data')
pert_data.load(data_path='./data/gse152988_crispri')

# Manual splits
all_perts = [p for p in pert_data.adata.obs['condition'].unique() if p != 'ctrl']
np.random.seed(1)
np.random.shuffle(all_perts)

n_total = len(all_perts)
pert_data.set2conditions = {
    'train': all_perts[:int(0.70*n_total)],
    'val': all_perts[int(0.70*n_total):int(0.85*n_total)],
    'test': all_perts[int(0.85*n_total):]
}

pert_data.split = 'custom'
pert_data.seed = 1
pert_data.train_gene_set_size = 0.75

print(f"Split: Train={len(pert_data.set2conditions['train'])}, Val={len(pert_data.set2conditions['val'])}, Test={len(pert_data.set2conditions['test'])}")

# Get dataloaders
pert_data.get_dataloader(batch_size=32, test_batch_size=128)

# Create empty graphs manually (valid but minimal)
num_genes = len(pert_data.gene_names)

# Empty edge indices (shape: [2, 0])
G_coexpress = torch.LongTensor(2, 0).to(device)
G_coexpress_weight = torch.Tensor(0).to(device)

# Train with pre-defined empty graphs
gears_model = GEARS(pert_data, device=device)
gears_model.model_initialize(
    hidden_size=64,
    G_coexpress=G_coexpress,
    G_coexpress_weight=G_coexpress_weight
)

gears_model.train(epochs=20)

Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
[]
Local copy of pyg dataset is detected. Loading...
Done!
Creating dataloaders....
Done!


Split: Train=128, Val=28, Test=28


Found local copy...
Start Training...
Epoch 1 Step 1 Train Loss: 8449996.0000
Epoch 1 Step 51 Train Loss: 16647491.0000
Epoch 1 Step 101 Train Loss: 15244825.0000
Epoch 1 Step 151 Train Loss: 32905640.0000
Epoch 1 Step 201 Train Loss: 6161983.5000
Epoch 1 Step 251 Train Loss: 42967692.0000
Epoch 1 Step 301 Train Loss: 22316476.0000
Epoch 1 Step 351 Train Loss: 3786040.7500
Epoch 1 Step 401 Train Loss: 13005115.0000
Epoch 1 Step 451 Train Loss: 7176703.0000
Epoch 1 Step 501 Train Loss: 4724781.0000
Epoch 1 Step 551 Train Loss: 44721804.0000
Epoch 1 Step 601 Train Loss: 10158371.0000
Epoch 1 Step 651 Train Loss: 14950472.0000
Epoch 1 Step 701 Train Loss: 17938066.0000
Epoch 1: Train Overall MSE: 0.1257 Validation Overall MSE: 0.1465. 
Train Top 20 DE MSE: 1.2613 Validation Top 20 DE MSE: 0.1648. 
Epoch 2 Step 1 Train Loss: 12941250.0000
Epoch 2 Step 51 Train Loss: 33526400.0000
Epoch 2 Step 101 Train Loss: 12630458.0000
Epoch 2 Step 151 Train Loss: 167642816.0000
Epoch 2 Step 201 Train L

In [ ]:
# set up and train a model
# gears_model = GEARS(pert_data, device=device)
# gears_model.model_initialize(hidden_size=64)
# gears_model.train(epochs=20)

## Step 5: Save Model

In [ ]:
# save model
gears_model.save_model('./models/gears_gse152988')

## Step 6.5: Evaluate Model Performance

In [ ]:
from gears.inference import evaluate, compute_metrics

In [ ]:
# Evaluate on held-out test perturbations
print("Evaluating on test set...")
test_res = evaluate(
    gears_model.dataloader['test_loader'],
    gears_model.model,
    gears_model.config['uncertainty'],
    gears_model.device
)

# Compute metrics
test_metrics, test_pert_res = compute_metrics(test_res)

print("\nTest Set Metrics:")
print(test_metrics)

Evaluating on test set...

Test Set Metrics:
{'mse': 0.10000738448330335, 'mse_de': 0.048754948490698426, 'pearson': 0.998632352585166, 'pearson_de': 0.9681612152705867}


In [ ]:
# Show results for each held-out perturbation
results_df = pd.DataFrame(test_pert_res).T
results_df = results_df.sort_values('pearson', ascending=False)

print(f"\nResults for {len(results_df)} held-out perturbations:")
print(results_df)


Results for 28 held-out perturbations:
               mse   pearson    mse_de  pearson_de
SCFD1     0.079112  0.999435  0.084143    0.987559
NSF       0.063037  0.999416  0.024545    0.989069
HEXA      0.108043  0.999392  0.030842    0.990062
RBM17     0.049381  0.999367  0.081143    0.946457
ECHS1     0.043868  0.999189  0.038498    0.992461
PDS5B     0.032823  0.999156  0.048266    0.985743
DYNC1H1   0.118880  0.999102  0.012062    0.970697
APEX1     0.041434  0.998980  0.166888    0.992813
BECN1     0.053984  0.998976  0.021742    0.987507
TRIP4     0.093548  0.998917  0.010721    0.964553
CNTNAP2   0.047011  0.998832  0.024833    0.985103
GSR       0.157803  0.998755  0.006601    0.996168
PARP1     0.045926  0.998730  0.045253    0.981662
NDUFV1    0.143446  0.998721  0.084041    0.874828
NCSTN     0.076953  0.998718  0.007195    0.861888
RHOT1     0.057356  0.998704  0.017991    0.993621
PFDN1     0.050975  0.998632  0.008979    0.926199
SCO2      0.430549  0.998499  0.016145    

## Step 7: Load Model

In [ ]:
# load model
# gears_model.load_pretrained('./models/gears_gse152988')
# Check if model from recent training exists in memory
try:
    # This will work if you just trained
    print("✓ Model exists in memory")
    print(f"Model device: {gears_model.device}")
    print(f"Model config exists: {gears_model.config is not None}")

    if gears_model.config:
        print(f"\nModel details:")
        print(f"  Hidden size: {gears_model.config['hidden_size']}")
        print(f"  Num genes: {gears_model.config['num_genes']}")
        print(f"  Trained on device: {gears_model.config['device']}")

    # Try a quick prediction to verify it works
    print("\n✓ Model is loaded and ready to save")

except NameError:
    print("✗ ERROR: No model in memory!")
    print("You need to run the training cell first")
except Exception as e:
    print(f"✗ ERROR: {e}")

✓ Model exists in memory
Model device: cuda:0
Model config exists: True

Model details:
  Hidden size: 64
  Num genes: 33538
  Trained on device: cuda:0

✓ Model is loaded and ready to save


## Step 8: Predict

In [ ]:
# predict (replace with genes from your dataset)
# gears_model.predict([['TAF1']])
# gears_model.GI_predict(['GENE1', 'GENE2'], GI_genes_file=None)